# CIC-IDS2017 feature selection

This notebook explains which network-flow features are most useful for separating benign traffic from attacks. Scores are calculated from training data only.

In [2]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULT_FOLDER = PROJECT_ROOT / "artifacts/feature_selection"

## Selection methods

- **Correlation:** measures the strength of a linear relationship with the binary label.
- **Mutual information:** detects both linear and nonlinear relationships.
- **Random Forest importance:** measures how much a feature helps the trees separate classes.
- **Combined score:** the mean of the three normalized scores.

In [3]:
report_file = RESULT_FOLDER / "selected_features.json"
score_file = RESULT_FOLDER / "feature_scores.csv"

if not report_file.exists():
    raise FileNotFoundError("Run 'uv run python select_features.py' first.")

report = json.loads(report_file.read_text(encoding="utf-8"))
scores = pd.read_csv(score_file).set_index("feature")

pd.Series({
    "Sample rows": report["sample_rows"],
    "Candidate features": report["candidate_features"],
    "Constant features removed": len(report["constant_features"]),
    "Selected features": report["selected_feature_count"],
}).to_frame("value")

,value
Sample rows,200000
Candidate features,69
Constant features removed,8
Selected features,20


## Constant features removed

A constant feature has the same value for every sampled flow, so it cannot help distinguish benign traffic from attacks.

In [4]:
pd.DataFrame({"constant_feature": report["constant_features"]})

,constant_feature
0,Bwd PSH Flags
1,Bwd URG Flags
2,Fwd Avg Bytes/Bulk
3,Fwd Avg Packets/Bulk
4,Fwd Avg Bulk Rate
5,Bwd Avg Bytes/Bulk
6,Bwd Avg Packets/Bulk
7,Bwd Avg Bulk Rate


## Top 20 features

In [ ]:
top_scores = scores.head(report["selected_feature_count"])
top_scores.style.background_gradient(subset=["combined_score"], cmap="Blues").format(precision=3)

In [ ]:
top_scores.sort_values("combined_score")["combined_score"].plot(
    kind="barh", figsize=(10, 8), color="steelblue"
)
plt.title("Top features by combined score")
plt.xlabel("Combined importance score")
plt.tight_layout()
plt.show()

## Compare the three methods

In [ ]:
method_columns = ["correlation", "mutual_information", "random_forest"]
top_scores.head(12)[method_columns].plot(kind="bar", figsize=(15, 6))
plt.title("Feature scores from each method")
plt.ylabel("Normalized score")
plt.xticks(rotation=55, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
scores[method_columns + ["combined_score"]].corr().style.background_gradient(
    cmap="coolwarm", vmin=-1, vmax=1
).format(precision=2)

## Conclusion

The selected features are saved in `selected_features.json`. The next stage should train Random Forest and Decision Tree models using only these columns. The untouched test set must only be used for final evaluation.